In [ ]:
import torch
import lightning
from mlcolvar.cvs import BaseCV
from mlcolvar.core import FeedForward
import numpy as np
from mlcolvar.data import DictDataset
from mlcolvar.cvs.committor.utils import initialize_committor_masses, compute_committor_weights
from mlcolvar.utils.io import create_dataset_from_files
from mlcolvar.cvs.generator import Generator

torch.set_default_dtype(torch.float64)

__all__ = ["Generator"]
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from mlcolvar.core.transform import Transform
from mlcolvar.core.transform.tools.utils import easy_KDE

class LogHistogram(Transform):
    """
    Compute continuous histogram using Gaussian kernels
    """

    def __init__(self,
                 in_features: int,
                 min: float,
                 max: float,
                 bins: int,
                 sigma_to_center: float = 1.0) -> torch.Tensor :
        """Computes the continuous histogram of a quantity using Gaussian kernels

        Parameters
        ----------
        in_features : int
            Number of inputs
        min : float
            Minimum value of the histogram
        max : float
            Maximum value of the histogram
        bins : int
            Number of bins of the histogram
        sigma_to_center : float, optional
            Sigma value in bin_size units, by default 1.0


        Returns
        -------
        torch.Tensor
            Values of the histogram for each bin
        """
       
        super().__init__(in_features=in_features, out_features=bins)

        self.min = min
        self.max = max
        self.bins = bins
        self.sigma_to_center = sigma_to_center
    
    def compute_hist(self, x):
        hist = easy_KDE(x=x,
                        n_input=self.in_features, 
                        min_max=[self.min, self.max], 
                        n=self.bins, 
                        sigma_to_center=self.sigma_to_center)
        return hist

    def forward(self, x: torch.Tensor):
        x = torch.log(self.compute_hist(x) + 1e-10) - -23.025850929940457  # add small value to avoid log(0)
        return x
    
from mlcolvar.core.transform import Transform
from mlcolvar.core.transform.tools.utils import easy_KDE

class LogHistogram32(Transform):
    """
    Compute continuous histogram using Gaussian kernels
    """

    def __init__(self,
                 in_features: int,
                 min: float,
                 max: float,
                 bins: int,
                 sigma_to_center: float = 1.0) -> torch.Tensor :
        """Computes the continuous histogram of a quantity using Gaussian kernels

        Parameters
        ----------
        in_features : int
            Number of inputs
        min : float
            Minimum value of the histogram
        max : float
            Maximum value of the histogram
        bins : int
            Number of bins of the histogram
        sigma_to_center : float, optional
            Sigma value in bin_size units, by default 1.0


        Returns
        -------
        torch.Tensor
            Values of the histogram for each bin
        """
       
        super().__init__(in_features=in_features, out_features=bins)

        self.min = min
        self.max = max
        self.bins = bins
        self.sigma_to_center = sigma_to_center
    
    def compute_hist(self, x):
        hist = easy_KDE(x=x,
                        n_input=self.in_features, 
                        min_max=[self.min, self.max], 
                        n=self.bins, 
                        sigma_to_center=self.sigma_to_center).to(torch.float32)
        return hist

    def forward(self, x: torch.Tensor):
        x = torch.log(self.compute_hist(x) + 1e-10) - -23.025850929940457  # add small value to avoid log(0)
        return x

In [2]:
#from mlcolvar.core.loss import GeneratorLoss,compute_eigenfunctions

class GeneratorLoss(torch.nn.Module):
    """
    Computes the loss to learn a representation for the generator
    """

    def __init__(self, eta, cell, friction, alpha, n_cvs,u_stat=True):
        """
        eta: float
        eta : float
          Hyperparameter for the shift to define the resolvent. $(\eta I-_mathcal{L})^{-1}$
        r : int
          Hyperparamer for the number of eigenfunctions wanted
        alpha : float
          Hyperparamer that scales the orthonormality loss
        cell : float
          CUBIC cell size length, used to scale the positions from reduce coordinates to real coordinates, by default None
        friction: torch.tensor
          Langevin friction which should contain \sqrt{k_B*T/(gamma*m_i)}
        n_cvs:
          useless, remove it
        """
        super().__init__()
        self.eta = eta
        self.friction = friction
        self.lambdas = torch.nn.Parameter(10 * torch.randn(n_cvs), requires_grad=True)
        self.alpha = alpha
        self.cell = cell
        self.u_stat=u_stat

    def forward(self, data, output, weights, gradient_descriptors=None):
        return generator_loss(
            data,
            output,
            self.eta,
            self.alpha,
            self.friction,
            self.cell,
            self.lambdas,
            weights,
            gradient_descriptors,
            u_stat=self.u_stat,
        )

# TODO check that maybe we can replace this by the one from deepTICA
def compute_covariance(X, weights):
    n = X.size(0)
    pre_factor = n / (n - 1)
    if X.ndim == 2:
        return pre_factor * (
            torch.einsum("ij,ik,i->jk", X, X, weights) / n
        )  # (X.T @ X / n - mean @ mean.T)
    else:
        return pre_factor * (torch.einsum("ijk,ilk,i->jl", X, X, weights) / n)


def generator_loss(
    data,
    output,
    eta,
    alpha,
    friction,
    cell,
    lambdas,
    weights,
    gradient_descriptors=None,
    u_stat=True
):
    """
    Computes the loss to learn the representation
    data: torch.tensor
      input of the NN
    output: torch.tensor
      output of the NN
    weights: torch.tensor
      Statistical weights
    gradient_descriptors: torch.tensor, optional
      Gradient of the descriptors with respect to the atomic positions.
      Used for the chain rule to compute the full gradients
    """
    lambdas = lambdas**2
    diag_lamb = torch.diag(lambdas)
    # sorted_lambdas = lambdas[torch.argsort(lambdas)]
    r = output.shape[1]
    sample_size = output.shape[0] // 2

    gradient = torch.stack(
        [
            torch.autograd.grad(
                outputs=output[:, idx].sum(),
                inputs=data,
                retain_graph=True,
                create_graph=True,
            )[0]
            for idx in range(r)
        ],
        dim=2,
    ).swapaxes(2, 1)
    ### jacrev seems to be a bit faster, but I need to pass the model as argument, and in order to go with the philosophy of the other losses I am keeping it like this for now
    # compute_batch_jacobian = functorch.vmap(functorch.jacrev(model,argnums=0),in_dims=(0))
    # gradient = compute_batch_jacobian(data.unsqueeze(1))
    gradient = gradient.reshape(weights.shape[0], output.shape[1], -1)

    if gradient_descriptors is None:  # If there is no descriptors or a precompute layer
        gradient_positions = gradient * torch.sqrt(friction)
    else:
        gradient_positions = torch.einsum(
            "ijk,imkl->ijml", gradient, gradient_descriptors
        )
        gradient_positions = gradient_positions.reshape(
            -1, output.shape[1], gradient_descriptors.shape[1] * gradient_descriptors.shape[-1]
        ) * torch.sqrt(friction)
    if cell is not None:
        gradient_positions /= cell
    ### In order to have unbiased estimation, we split the dataset in two chunks ###
    if u_stat:
        weights_X, weights_Y = weights[:sample_size], weights[sample_size:]
        gradient_X, gradient_Y = (
            gradient_positions[:sample_size],
            gradient_positions[sample_size:],
        )
        psi_X, psi_Y = output[:sample_size], output[sample_size:]

        cov_X = compute_covariance(psi_X, weights_X)

        cov_Y = compute_covariance(psi_Y, weights_Y)

        dcov_X = compute_covariance(gradient_X, weights_X)

        dcov_Y = compute_covariance(gradient_Y, weights_Y)

        W1 = (eta * cov_X + dcov_X) @ diag_lamb
        W2 = (eta * cov_Y + dcov_Y) @ diag_lamb

    ### Unbiased estimation of the "variational part"
    # It might be worse replacing with einsum if it is faster
        loss_ef = torch.trace(
            ((cov_X @ diag_lamb) @ W2 + (cov_Y @ diag_lamb) @ W1) / 2
            - cov_X @ diag_lamb
            - cov_Y @ diag_lamb
        )

    # Compute loss_ortho
        loss_ortho = alpha * (
            torch.trace(
                (torch.eye(output.shape[1], device=output.device) - cov_X).T
                @ (torch.eye(output.shape[1], device=output.device) - cov_Y)
            )
        )
    else:

        cov = compute_covariance(output, weights)

        dcov = compute_covariance(gradient_positions, weights)


        W = (eta * cov + dcov) @ diag_lamb

    ### Unbiased estimation of the "variational part"
    # It might be worse replacing with einsum if it is faster
        loss_ef = torch.trace(
            (cov @ diag_lamb) @ W
            - 2*cov @ diag_lamb
        )

    # Compute loss_ortho
        loss_ortho = alpha * (
            torch.trace(
                (torch.eye(output.shape[1], device=output.device) - cov).T
                @ (torch.eye(output.shape[1], device=output.device) - cov)
            )
        )
    loss = loss_ef + loss_ortho  # loss_ortho
    return loss, loss_ef, loss_ortho

def compute_eigenfunctions(
    input,
    output,
    weights,
    friction,
    eta,
    r,
    cell=None,
    tikhonov_reg=1e-4,
    descriptors_derivatives=None,
):
    """
    Computes eigenfunctions and eigenvalues from a learned representation.

    This function estimates the eigenfunctions and eigenvalues of the infinitesimal generator
    associated with the Langevin process. The eigenvalues are computed using a resolvent approach,
    where `evals` relate to the generator's eigenvalues as: `lambda = eta - 1/evals`.

    Parameters:
    ----------
    input : torch.Tensor, shape (N,d)
        Input tensor containing the data
    output : torch.Tensor, shape (N,r)
        Output tensor containing the learned representation of the data
    weights : torch.Tensor, shape (N,)
        Biasing weights (set to one for unbiased simulations)
    friction : torch.Tensor, shape (N,)
        Langevin friction values for each data point.
    eta : float
        Hyperparameter for the resolvent approach.
    r : int
        Number of eigenfunctions to compute.

    cell : torch.Tensor, optional, shape (3,3)
        If provided, used to normalize the gradients when periodic boundary conditions apply.
    descriptors_derivatives : torch.Tensor, optional, shape (N, natoms, d, n_spatial)
        Derivatives of descriptors with respect to atomic positions. If `None`,
        the function uses direct gradients of `model(X)`.
    Returns:
    --------
    g : torch.Tensor, shape (N, r)
        The computed eigenfunctions evaluated at each data point.
    lambdas : torch.Tensor, shape (r,)
        The eigenvalues associated with the generator, sorted in descending order.
    evecs : torch.Tensor, shape (r, r)
        The eigenvectors of the operator.
    Notes:
    ------
    - Eigenfunctions are normalized using the dataset weights.
    - If `gradient_descriptors` is provided, the function projects the gradients onto it.
    - The operator matrix is regularized to improve numerical stability.
    """
    # friction=friction.to("cuda")

    d = input.shape[1]

    # gradient with respect to descriptors
    gradient_X = torch.stack(
        [
            torch.autograd.grad(
                outputs=output[:, idx].sum(),
                inputs=input,
                retain_graph=True,
                create_graph=True,
            )[0].reshape((-1, d))
            for idx in range(r)
        ],
        dim=2,
    ).swapaxes(2, 1)

    if descriptors_derivatives is not None:  # If there is no precomputing layer
        gradient_positions = torch.einsum(
            "ijk,imkl->ijml", gradient_X, descriptors_derivatives
        )
        gradient_positions = gradient_positions.reshape(
            -1, output.shape[1], descriptors_derivatives.shape[1] * descriptors_derivatives.shape[-1]
        ) * torch.sqrt(friction)

    else:
        gradient_positions = gradient_X * torch.sqrt(friction)
    if cell is not None:
        gradient_positions /= cell

    weights_X = weights

    # Compute covariances
    cov_X = compute_covariance(output, weights_X)
    dcov_X = compute_covariance(gradient_positions, weights_X)

    W = eta * cov_X + dcov_X
    # The resolvent projected on the learned space
    operator = (
        torch.linalg.inv(
            W + tikhonov_reg * torch.eye(output.size(1), device=output.device)
        )
        @ cov_X
    )
    evals, evecs = torch.linalg.eig(operator)
    # eigenfunctions
    g = output @ evecs.real
    lambdas = eta - 1 / evals  # eigenvalues of the generator
    sorting = torch.argsort(-lambdas.real)
    # Ensure normalization of eigenfunctions
    detached_evecs = evecs.detach()
    detached_evecs /= torch.sqrt(torch.mean(weights.unsqueeze(1) * g**2, axis=0))
    g /= torch.sqrt(torch.mean(weights.unsqueeze(1) * g**2, axis=0))
    return g[:, sorting], lambdas.detach()[sorting], detached_evecs.detach()[:, sorting]


class Generator_activation(BaseCV, lightning.LightningModule):
    """
    Baseclass for learning a representation for the eigenfunctions of the generator.
    The representation is expressed as a concatenation of the output of r neural networks.
    **Data**: for training it requires a DictDataset with the keys 'data', and 'weights'
              and optionally 'derivatives' which should contain the descriptors derivatives
    **Loss**: Minimize the representation loss and the orthonormalization loss

    """

    BLOCKS = ["nn"]

    def __init__(
        self,
        layers: list,
        eta: float,
        r: int,
        alpha: float,
        friction=None,
        cell: float = None,
        options: dict = None,
        **kwargs,
    ):
        """Define a NN-based generator model

        Parameters
        ----------
        layers : list
            Number of neurons per layer
        eta : float
            Hyperparameter for the shift to define the resolvent. $(\eta I-_mathcal{L})^{-1}$
        r : int
            Hyperparamer for the number of eigenfunctions wanted
        alpha : float
            Hyperparamer that scales the orthonormality loss
        friction: torch.tensor
            Langevin friction which should contain \sqrt{k_B*T/(gamma*m_i)}
        cell : float, optional
            CUBIC cell size length, used to scale the positions from reduce coordinates to real coordinates, by default None

        options : dict[str, Any], optional
            Options for the building blocks of the model, by default {}.
            Available blocks: ['nn'] .
        """
        super().__init__(in_features=layers[0], out_features=r, **kwargs)

        # =======  LOSS  =======
        self.loss_fn = GeneratorLoss(
            eta=eta, alpha=alpha, cell=cell, friction=friction, n_cvs=r
        )
        self.r = 1
        self.eta = eta
        self.friction = friction
        self.cell = cell
        self.evecs = None
        self.evals = None
        # ======= OPTIONS =======
        # parse and sanitize
        options = self.parse_options(options)

        # ======= BLOCKS =======
        # initialize NN turning
        o = "nn"
        # set default activation to tanh
        if "activation" not in options[o]:
            options[o]["activation"] = "tanh"
        self.nn = FeedForward(layers, **options[o])

    def compute_eigenfunctions(
        self,
        dataset,
        friction=None,
        eta=None,
        cell=None,
        tikhonov_reg=1e-4,
        recompute=False,
    ):
        """Computes the eigenfunctions based on the representation learned given by the neural networks.

        Parameters
        ----------
        dataset : DictDataset
        Dictionary containing:
            - 'data' (torch.Tensor, shape (N, d)): Input descriptors or positions.
            - 'weights' (torch.Tensor, shape (N,)): Biasing weights associated with the data points.
            - 'derivatives', optional, (torch.Tensor, shape (N,natoms, d, n_spatial)): derivatives of the descriptors with respect to the atomic positions
        friction:torch.tensor, optional
            If different from the one used in training: Langevin friction which should contain \sqrt{k_B*T/(gamma*m_i)}
        eta : float, optional
            If different from the one used in training, Hyperparameter for the shift to define the resolvent. $(\eta I-_mathcal{L})^{-1}$

        cell : float, optional
            If different form the one used in training, CUBIC cell size length, used to scale the positions from reduce coordinates to real coordinates, by default None

        tikhonov_reg: float, optional
            Hyperparameter for the regularization of the inverse (Ridge regression parameter)
        recompute: Boolean, optional
            Is used to know if the eigenvectors are needed to be recomputed or not
        """
        if friction is None:
            friction = self.friction
        if eta is None:
            eta = self.eta
        if cell is None:
            cell = self.cell
        if (
            recompute or self.evecs is None
        ):  # If the calculation has not been done previously, or we want to compute again the eigenpairs due to a change of parameters
            dataset["data"].requires_grad = True
            output = self.forward(dataset["data"])
            if "derivatives" in dataset.keys:
                descriptors_derivatives = dataset["derivatives"]
            else:
                descriptors_derivatives = None
            eigenfunctions, evals, evecs = compute_eigenfunctions(
                dataset["data"],
                output,
                dataset["weights"],
                friction,
                eta,
                self.r,
                cell,
                tikhonov_reg,
                descriptors_derivatives=descriptors_derivatives,
            )
            self.evals = evals
            self.evecs = evecs
            return eigenfunctions, evals, evecs

        else:
            eigenfunctions = self.forward(dataset["data"]) @ self.evecs.real
            return eigenfunctions, self.evals, self.evecs

    def forward_cv(self, x: torch.Tensor) -> torch.Tensor:

        return torch.exp(-self.nn(x))

    def training_step(self, train_batch, batch_idx):
        """Compute and return the training loss and record metrics."""
        torch.set_grad_enabled(True)
        # =================get data===================
        x = train_batch["data"]
        # check data are have shape (n_data, -1)
        x = x.reshape((x.shape[0], -1))

        x.requires_grad = True

        weights = train_batch["weights"]
        if "derivatives" in train_batch.keys():
            derivatives = train_batch["derivatives"]
        else:
            derivatives = None

        # =================forward====================
        # we use forward and not forward_cv to also apply the preprocessing (if present)
        q = self.forward(x)
        # ===================loss=====================
        if self.training:
            loss, loss_ef, loss_ortho = self.loss_fn(x, q, weights, derivatives)
        else:
            loss, loss_ef, loss_ortho = self.loss_fn(x, q, weights, derivatives)
        # ====================log=====================+
        name = "train" if self.training else "valid"
        self.log(f"{name}_loss", loss, on_epoch=True)
        self.log(f"{name}_loss_var", loss_ef, on_epoch=True)
        self.log(f"{name}_loss_ortho", loss_ortho, on_epoch=True)
        return loss


In [4]:
import torch
import lightning
from mlcolvar.cvs import BaseCV
from mlcolvar.core import FeedForward
#from mlcolvar.core.loss.generator_loss import GeneratorLoss
from mlcolvar.core.loss.committor_loss import SmartDerivatives, compute_descriptors_derivatives

__all__ = ["Generator"]


class ghostCV_combi_trivial(BaseCV, lightning.LightningModule):
    """Base class for data-driven learning of committor function.
    The committor function q is expressed as the output of a neural network optimized with a self-consistent
    approach based on the Kolmogorov's variational principle for the committor and on the imposition of its boundary conditions. 

    **Data**: for training it requires a DictDataset with the keys 'data', 'labels' and 'weights'

    **Loss**: Minimize Kolmogorov's variational functional of q and impose boundary condition on the metastable states (CommittorLoss)
    
    References
    ----------
    .. [*] P. Kang, E. Trizio, and M. Parrinello, "Computing the committor using the committor to study the transition state ensemble", Nat. Comput. Sci., 2024, DOI: 10.1038/s43588-024-00645-0

    See also
    --------
    mlcolvar.core.loss.CommittorLoss
        Kolmogorov's variational optimization of committor and imposition of boundary conditions
    mlcolvar.cvs.committor.utils.compute_committor_weights
        Utils to compute the appropriate weights for the training set
    mlcolvar.cvs.committor.utils.initialize_committor_masses
        Utils to initialize the masses tensor for the training
    """

    BLOCKS = ["nn", "sigmoid"]

    def __init__(
        self, 
        layers: list,
        eta: float,
        r: int,
        alpha: float = 10000,
        cell: float = None,
        friction = None,
        options: dict = None,
        coeffs = None,
        min =None,
        max =None,
        **kwargs,
    ):
        """Define a NN-based committor model

        Parameters
        ----------
        layers : list
            Number of neurons per layer
        mass : torch.Tensor
            List of masses of all the atoms we are using, for each atom we need to repeat three times for x,y,z.
            The mlcolvar.cvs.committor.utils.initialize_committor_masses can be used to simplify this.
        alpha : float
            Hyperparamer that scales the boundary conditions contribution to loss, i.e. alpha*(loss_bound_A + loss_bound_B)
        gamma : float, optional
            Hyperparamer that scales the whole loss to avoid too small numbers, i.e. gamma*(loss_var + loss_bound), by default 10000
        delta_f : float, optional
            Delta free energy between A (label 0) and B (label 1), units is kBT, by default 0. 
            State B is supposed to be higher in energy.
        cell : float, optional
            CUBIC cell size length, used to scale the positions from reduce coordinates to real coordinates, by default None
        options : dict[str, Any], optional
            Options for the building blocks of the model, by default {}.
            Available blocks: ['nn'] .
        """
        super().__init__(in_features=layers[0], out_features=layers[-1], **kwargs) 
        
        # =======  LOSS  =======
        self.loss_fn = GeneratorLoss(
                                     eta=eta,
                                     alpha=alpha,
                                     cell=cell,
                                     friction=friction,
                                     n_cvs=r
        )

        # ======= OPTIONS =======
        # parse and sanitize
        options = self.parse_options(options)

        # ======= BLOCKS =======
        # initialize NN turning
        o = "nn"
        # set default activation to tanh
        if "activation" not in options[o]: 
            options[o]["activation"] = "tanh"
        self.nn = FeedForward(layers, **options[o])
        self.coeffs = coeffs
        print(min)

        
    def forward_cv(self, x: torch.Tensor) -> (torch.Tensor):
        output = self.nn(x)
        return torch.exp(-output)

class ghostCV_combi(BaseCV, lightning.LightningModule):
    """Base class for data-driven learning of committor function.
    The committor function q is expressed as the output of a neural network optimized with a self-consistent
    approach based on the Kolmogorov's variational principle for the committor and on the imposition of its boundary conditions. 

    **Data**: for training it requires a DictDataset with the keys 'data', 'labels' and 'weights'

    **Loss**: Minimize Kolmogorov's variational functional of q and impose boundary condition on the metastable states (CommittorLoss)
    
    References
    ----------
    .. [*] P. Kang, E. Trizio, and M. Parrinello, "Computing the committor using the committor to study the transition state ensemble", Nat. Comput. Sci., 2024, DOI: 10.1038/s43588-024-00645-0

    See also
    --------
    mlcolvar.core.loss.CommittorLoss
        Kolmogorov's variational optimization of committor and imposition of boundary conditions
    mlcolvar.cvs.committor.utils.compute_committor_weights
        Utils to compute the appropriate weights for the training set
    mlcolvar.cvs.committor.utils.initialize_committor_masses
        Utils to initialize the masses tensor for the training
    """

    BLOCKS = ["nn", "sigmoid"]

    def __init__(
        self, 
        layers: list,
        eta: float,
        r: int,
        alpha: float = 10000,
        cell: float = None,
        friction = None,
        options: dict = None,
        coeffs = None,
        min =None,
        max =None,
        **kwargs,
    ):
        """Define a NN-based committor model

        Parameters
        ----------
        layers : list
            Number of neurons per layer
        mass : torch.Tensor
            List of masses of all the atoms we are using, for each atom we need to repeat three times for x,y,z.
            The mlcolvar.cvs.committor.utils.initialize_committor_masses can be used to simplify this.
        alpha : float
            Hyperparamer that scales the boundary conditions contribution to loss, i.e. alpha*(loss_bound_A + loss_bound_B)
        gamma : float, optional
            Hyperparamer that scales the whole loss to avoid too small numbers, i.e. gamma*(loss_var + loss_bound), by default 10000
        delta_f : float, optional
            Delta free energy between A (label 0) and B (label 1), units is kBT, by default 0. 
            State B is supposed to be higher in energy.
        cell : float, optional
            CUBIC cell size length, used to scale the positions from reduce coordinates to real coordinates, by default None
        options : dict[str, Any], optional
            Options for the building blocks of the model, by default {}.
            Available blocks: ['nn'] .
        """
        super().__init__(in_features=layers[0], out_features=layers[-1], **kwargs) 
        
        # =======  LOSS  =======
        self.loss_fn = GeneratorLoss(
                                     eta=eta,
                                     alpha=alpha,
                                     cell=cell,
                                     friction=friction,
                                     n_cvs=r
        )

        # ======= OPTIONS =======
        # parse and sanitize
        options = self.parse_options(options)

        # ======= BLOCKS =======
        # initialize NN turning
        o = "nn"
        # set default activation to tanh
        if "activation" not in options[o]: 
            options[o]["activation"] = "tanh"
        self.nn =FeedForward(layers, **options[o]) 
        self.coeffs = coeffs
        print(min)

        
    def forward_cv(self, x: torch.Tensor) -> (torch.Tensor):
        output = torch.exp(-self.nn(x))#torch.cat([nn(x) for nn in self.nn], dim=1)
        return output

def convert_model(model_name, n_input):
    loaded_model = torch.jit.load(model_name).to(torch.device('cpu')).to(torch.float32)
    fake_input = torch.rand(1,n_input,dtype=torch.float32).to(torch.device('cpu')).to(torch.float32)
    loaded_model(fake_input)
    frozen_model = torch.jit.trace(loaded_model, fake_input)
    torch.jit.save(frozen_model, model_name)
class BiasModel(torch.nn.Module):
    def __init__(self, input_model, e=1e-6, l=1) -> None:
        super().__init__()
        self.input_model = input_model
        self.l = l
        if type(e) is not torch.Tensor:
            e = torch.tensor([e],dtype=torch.float32,device=torch.device("cpu"))
        self.e = e.to("cpu")

    def forward(self, x):
        x.requires_grad = True
        q = self.input_model(x)
        print(q.device)
        grad_outputs = torch.ones_like(q,device="cpu",dtype=torch.float32)
        grads = torch.autograd.grad(q, x, grad_outputs, retain_graph=True)[0]
        grads_squared = torch.sum(torch.pow(grads, 2), 1)
        bias =  -self.l* (torch.log( grads_squared + self.e ) - torch.log(self.e))
        return bias

def create_and_save_ghost(model, layers, r, friction, coeffs,iteration ):
    new_model = ghostCV_combi(layers=layers,eta=0.1,r=r,cell=None,alpha=200,friction=friction,coeffs=coeffs).to("cpu")
    new_model_trivial = ghostCV_combi_trivial(layers=layers,eta=0.1,r=r,cell=None,alpha=200,friction=friction,coeffs=coeffs[:,0]).to("cpu")
    new_model.nn = model.nn.to("cpu")
    new_model_trivial.nn = model.nn.to("cpu")
    new_model.to(torch.float32)
    new_model_trivial.to(torch.float32)
    new_model.preprocessing = LogHistogram32(in_features=32, min=0, max=10, bins=20)
    new_model_trivial.preprocessing = LogHistogram32(in_features=32, min=0, max=10, bins=20)
    traced_model = new_model.to("cpu").to_torchscript(file_path=f'models/model_{iteration}.pt', method='trace')
    traced_model = new_model_trivial.to("cpu").to_torchscript(file_path=f'models/model_{iteration}_trivial.pt', method='trace')

    convert_model(f'models/model_{iteration}.pt',32)
    convert_model(f'models/model_{iteration}_trivial.pt',32)
    return new_model, new_model_trivial
class BiasModel(torch.nn.Module):
    def __init__(self, input_model, e=1e-6, l=1) -> None:
        super().__init__()
        self.input_model = input_model
        self.l = l
        if type(e) is not torch.Tensor:
            e = torch.tensor([e],dtype=torch.float32,device=torch.device("cpu"))
        self.e = e.to("cpu")

    def forward(self, x):
        x.requires_grad = True
        q = self.input_model(x)
        print(q.device)
        grad_outputs = torch.ones_like(q,device="cpu",dtype=torch.float32)
        grads = torch.autograd.grad(q, x, grad_outputs, retain_graph=True)[0]
        grads_squared = torch.sum(torch.pow(grads, 2), 1)
        bias =  -self.l* (torch.log( grads_squared + self.e ) - torch.log(self.e))
        return bias

def load_data(filenames, regexp, load_args, index_iteration, beta, ComputeDescriptors, n_atoms):
    dataset, dataframe = create_dataset_from_files(file_names = filenames,
                                               folder = None,
                                               create_labels = True,
                                               filter_args={'regex' : regexp},
                                               return_dataframe = True,
                                               load_args=load_args,
                                               verbose = True)

    bias = torch.zeros(len(dataset))
    dataset = compute_committor_weights(dataset, bias, list(range(index_iteration+1)), beta)    
    pos, desc, d_desc_d_pos = compute_descriptors_derivatives(dataset, ComputeDescriptors, n_atoms, separate_boundary_dataset = False)
    dataset = DictDataset({"data":desc.clone().detach().to(device), "weights":dataset["weights"].to(device),"derivatives":d_desc_d_pos.clone().detach().to(device)})#30 2500 epochs
    return dataset, dataframe

In [3]:
#cell = torch.Tensor([3.0233, 3.0233, 3.0233]).to(device)
#print('Cell: ', cell)
# temperature in Kelvin
T = 1
# Boltzmann factor in the RIGHT ENRGY UNITS!
kb = 1
beta = 1/(kb*T)
kT = 1/beta
n_atoms = 32
print(f'Beta: {beta} \n1/beta: {1/beta}')
atomic_masses = initialize_committor_masses(atom_types=[0]*n_atoms, masses=[1])

Beta: 1.0 
1/beta: 1.0


In [5]:
filenames = ['../../Paper/321_well/unbiased/A/COLVAR',
            ]

load_args = [{'start' : 0, 'stop': 1000, 'stride': 1},
            ]           

# #######################################################################################

dataset, dataframe = create_dataset_from_files(file_names=filenames,
                                               create_labels=True,
                                               filter_args={'regex': 'p.x'}, # to load many positions --> 'regex': 'p[1-9]\.[abc]|p[1-2][0-9]\.[abc]'
                                               return_dataframe=True,
                                               load_args=load_args,
                                               verbose=True)

Class 0 dataframe shape:  (1000, 9)

 - Loaded dataframe (1000, 9): ['time', 'p.x', 'p.y', 'p.z', 'ene', 'pot.bias', 'pot.ene_bias', 'walker', 'labels']
 - Descriptors (1000, 1): ['p.x']


In [6]:
# Create synthetic data for 40 unbiased replicas
n_atoms = 32
new_data = torch.zeros(1000, n_atoms)
old_A = dataset["data"]

for i in range(n_atoms):
    new_data[:,i:i+1] = old_A[torch.randperm(old_A.size()[0])]

In [7]:
dataset["data"] = new_data

In [8]:
# Define the descriptors
from mlcolvar.core.transform import Transform

# Implement Three Two One energy here
class ThreeTwoOneEnergy(Transform):
    """
    Energy of given an atoms position
    """

    def __init__(self, n_atoms) -> torch.Tensor:
        """
        n_atoms: number of atoms in the system
        """
        super().__init__(in_features=1, out_features=1)
        self.n_atoms = n_atoms

    def compute_energy(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(-1, self.n_atoms, 1)
        x = x[:, :, 0]
        ene = 160*x**4-80*x**2+1.1221*x+10.563
        return ene.view(-1,self.n_atoms)

    def forward(self, x: torch.Tensor):
        ene = self.compute_energy(x)
        return ene

In [10]:
energy = ThreeTwoOneEnergy(n_atoms=n_atoms)

hist = LogHistogram(in_features=n_atoms, min=0, max=10, bins=20)
preprocessing = torch.nn.Sequential(energy, hist)

In [11]:
from mlcolvar.data import DictModule, DictDataset
#from mlcolvar.core.loss.committor_loss import compute_descriptors_derivatives, SmartDerivatives
from mlcolvar.cvs.committor.utils import compute_committor_weights

# compute weights
ds = compute_committor_weights(dataset=dataset,
                                    bias=torch.zeros(len(dataset)),
                                    data_groups=[0],
                                    beta=beta)

In [12]:
from mlcolvar.core.transform.descriptors.utils import sanitize_positions_shape

def compute_descriptors_derivatives(dataset, 
                                    descriptor_function, 
                                    n_atoms : int, 
                                    separate_boundary_dataset = True, 
                                    positions_noise : float = 0.0,
                                    batch_size : int = None):
    """Compute the derivatives of a set of descriptors wrt input positions in a dataset for committor optimization

    Parameters
    ----------
    dataset :
        DictDataset with the positions under the 'data' key
    descriptor_function : torch.nn.Module
        Transform module for the computation of the descriptors
    n_atoms : int
        Number of atoms in the system
    separate_boundary_dataset : bool, optional
            Switch to exculde boundary condition labeled data from the variational loss, by default True
    positions_noise : float
        Order of magnitude of small noise to be added to the positions to avoid atoms having the exact same coordinates on some dimension and thus zero derivatives, by default 0.
        Ideally the smaller the better, e.g., 1e-6 for single precision, even lower for double precision.
    batch_size : int
        Size of batches to process data, useful for heavy computation to avoid memory overflows, if None a singel batch is used, by default None 

    Returns
    -------
    pos : torch.Tensor
        Positions tensor (detached)
    desc : torch.Tensor
        Computed descriptors (detached)
    d_desc_d_pos : torch.Tensor
        Derivatives of desc wrt to pos, shape (N, n_atoms*n_spatial, n_desc, 1) (detached)
    """
    
    # apply noise if given
    if positions_noise > 0:
        noise = torch.rand_like(dataset['data'], )*positions_noise
        dataset['data'] = dataset['data'] + noise

    # get and prepare positions
    pos = dataset['data']
    labels = dataset['labels']
    #pos = sanitize_positions_shape(pos=pos, n_atoms=n_atoms)[0]
    pos.requires_grad = True
    
    # get_device 
    device = pos.device

    # check if to separate boundary data
    if separate_boundary_dataset:
        mask_var = labels.squeeze() > 1
        if mask_var.sum()==0:
            raise(ValueError('No points left after separating boundary and variational datasets. \n If you are using only unbiased data set separate_boundary_dataset=False here and in Committor or don\'t use SmartDerivatives!!'))
    else:
        mask_var = torch.ones_like(labels.squeeze()).to(torch.bool)
    
    # check batches size for calculation
    if batch_size is None or batch_size == -1:
        batch_size = len(pos)
    else:
        if batch_size <= 0:
            raise ( ValueError(f"Batch size must be larger than zero if set! Found {batch_size}"))
    n_batches = int(np.ceil(len(pos) / batch_size))

    # compute descriptors and derivatives
    # we loop over batches and compute everything only for that part of the data, inside we loop over descriptors
    # we save lists and make them proper tensors later
    batch_aux_stack = []
    batch_desc_stack = []
    batch_count = 0
    while batch_count * batch_size + 1 <= len(pos):
        print(f"Processing batch {batch_count}/{n_batches}", end='\r')

        # get batch slicing indexes, they don't need to be all of the same size
        batch_start, batch_stop = batch_count*batch_size, (batch_count+1) * batch_size
        
        batch_mask_var = mask_var[batch_start:batch_stop]   # separate_dataset mask
        batch_pos = pos[batch_start:batch_stop]             # batch positions
        batch_pos = batch_pos[batch_mask_var, :]         # batch_positions for variational dataset only
        
        if len(batch_pos) > 0:
            batch_desc = descriptor_function(batch_pos)

            # loop over descriptors, #TODO maybe can be done with jacobians?
            # we store things always on the cpu
            batch_aux = []
            for i in range(len(batch_desc[0])):
                aux_der = torch.autograd.grad(batch_desc[:,i], batch_pos, grad_outputs=torch.ones_like(batch_desc[:,i]), retain_graph=True )[0]
                batch_aux.append(aux_der.detach().cpu())
            
            batch_d_desc_d_pos = torch.stack(batch_aux, axis=2)         # derivatives of this batch: (batch, n_positions, n_desc)
            batch_aux_stack.append(batch_d_desc_d_pos.detach().cpu())   # derivatives of all batches
            batch_desc_stack.append(batch_desc.detach().cpu())         # descriptors of all batches

            # cleanup
            del aux_der    
            del batch_pos
            del batch_desc

            # to be sure, clean the gpu cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        batch_count += 1
    
    print(f"Processed all data in {n_batches} batches!")

    if batch_count == 1:
        d_desc_d_pos = batch_d_desc_d_pos
        desc = batch_desc_stack
    else:
        d_desc_d_pos = torch.cat(batch_aux_stack, dim=0)
        desc = torch.cat(batch_desc_stack, dim=0)
    
    # we compute the descriptors on the whole dataset to always have all of them, no need for grads   
    #with torch.no_grad():
    #    print(pos.shape)
    #    desc = descriptor_function(pos)

    # detach and move back to original device
    pos = pos.detach().to(device)
    desc = desc.detach().to(device)
    d_desc_d_pos = d_desc_d_pos.detach().to(device)

    # to be sure, clean the gpu cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # unsqueeze(-1) adds the spatial dimension: (N, n_positions, n_desc) -> (N, n_positions, n_desc, n_spatial)
    # For 1D positions n_spatial=1, for 3D flat positions n_spatial=1 (already flattened into n_positions).
    return pos, desc, d_desc_d_pos.unsqueeze(-1)


In [15]:
pos, desc, d_desc_d_pos = compute_descriptors_derivatives(ds, preprocessing, n_atoms, separate_boundary_dataset = False, batch_size=103)
dataset = DictDataset({"data":desc.clone().detach().to(device), "weights":ds["weights"].to(device),"derivatives":d_desc_d_pos.clone().detach().to(device)})#30 2500 epochs

Processed all data in 10 batches!


In [16]:
gamma = 1/0.05
friction = np.zeros(n_atoms)
print(friction.shape)
for i_atom in range(n_atoms):
    friction[i_atom:i_atom+1] = np.array([kT / (gamma*atomic_masses[i_atom])]) 

friction = torch.tensor(friction, device=device,dtype=torch.float32)

(32,)


In [ ]:
from mlcolvar.data import DictModule
from mlcolvar.utils.io import create_dataset_from_files
from mlcolvar.cvs.committor.utils import compute_committor_weights
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from mlcolvar.utils.trainer import MetricsCallback
from mlcolvar.utils.plot import plot_metrics, paletteFessa, paletteCortina
#from mlcolvar.core.loss.generator_loss import compute_eigenfunctions
import os
import gc
import matplotlib.pyplot as plt
import MDAnalysis as mda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seeds = range(0,5,1)
options = { 'nn':{'activation':'tanh'},
         'optimizer' : {'lr': 5e-4, 'weight_decay': 1e-5}, 
           }  
for i in seeds:
    torch.manual_seed(i)
    
    # Detach dataset tensors to ensure they are leaf variables for each new model
    dataset_iter = DictDataset({
        "data": dataset["data"].detach().clone(),
        "weights": dataset["weights"].detach().clone(),
        "derivatives": dataset["derivatives"].detach().clone()
    })
    
    model = Generator_activation(layers=[20,32,32,1],eta=0.05,r=1,alpha=20,friction=friction, options=options)

    datamodule = DictModule(dataset_iter, lengths=[0.8,0.2])
    #wandb_logger = WandbLogger(name=f"seed_{i}_activation",project='dipeptide_clean')
    metrics = MetricsCallback()

    early_stop_callback = EarlyStopping(monitor="train_loss", min_delta=1e-4, patience=500, verbose=False)

    trainer = lightning.Trainer(callbacks=[metrics],#,early_stop_callback], 
                            max_epochs=2000, 
                            enable_checkpointing=False,
                            logger=False,
                            limit_val_batches=0,    # this to skip validation
                            num_sanity_val_steps=0  # this to skip validation
                            )
    # fit model
    trainer.fit(model, datamodule)

    torch.save(model,f"models/model_iter_{i}_all.pt")
    fig, ax = plt.subplots(1,1,figsize=(4,3))
    ax = plot_metrics(metrics.metrics,
                  keys=['train_loss', 'train_loss_var', 'train_loss_ortho'],
                  colors=['fessa1', 'fessa3', 'fessa4', 'fessa5'],
                  ax = ax, yscale="log")

    model = model.to(device)
    g, evals, evecs = model.compute_eigenfunctions(dataset_iter)

    coeffs = evecs.cpu().detach().real
    new_model_all, new_model = create_and_save_ghost(model, [20,32,32,1], 1, friction, coeffs, i )


    bias = BiasModel(model.to("cpu"),l=1,e=1e-7).to(torch.float64)
    lambda_value = 20/(bias(dataset_iter["data"].cpu().detach()).max() - bias(dataset_iter["data"].cpu().detach()).min())
    value = lambda_value * bias(dataset_iter["data"].cpu().detach()).max()
    print("lambda_value", lambda_value)
    print("value", value)

    
    # === CLEANUP CUDA MEMORY ===
    # Move model to CPU before deleting
    model = model.to("cpu")
    
    # Delete references to objects holding GPU memory
    del model, trainer, datamodule, dataset_iter, metrics
    del g, evals, evecs, coeffs
    del new_model_all, new_model, bias
    
    # Run garbage collection
    gc.collect()
    
    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    # Close the figure to free matplotlib memory
    plt.close(fig)